# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All Croissant entities such as record sets, fields, and columns are referenced by their `@id` values.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Instantiate dataset
dataset = mlc.Dataset(croissant_url)

## Access metadata (as a Python dict for display)
metadata_dict = json.loads(dataset.metadata.to_json_ld())

print("Dataset Name: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)
print("Published: ", getattr(dataset.metadata, 'datePublished', 'N/A'))

## 2. Data Overview

We review available record sets, fields, and their `@id` values.

Below, you will see a summary of the record sets (tables) found in the dataset, with their metadata and constituent fields, each referenced by their `@id`.

In [ ]:
# List available record sets and their field @ids

record_sets = dataset.metadata.recordSets

print("Found Record Sets:")
record_set_ids = []
for rs in record_sets:
    print(f"  - Record set @id: {rs['@id']}")
    print(f"    Name: {rs.get('name', 'N/A')}")
    print(f"    Description: {rs.get('description', 'No description')}")
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      · {field['@id']} (name: {field.get('name','?')}) type: {field.get('dataType', 'N/A')}")
    record_set_ids.append(rs['@id'])
print("\nAll Record Set @ids:", record_set_ids)

# For demonstration, preview the first record from each record set (if any)
for rs_id in record_set_ids:
    print(f"First record from record set {rs_id}:")
    for record in dataset.records(record_set=rs_id):
        print(record)
        break
    print()

## 3. Data Extraction

We will now load each record set identified above into a pandas DataFrame. All extractions are referenced explicitly by Croissant `@id` as per best practice.

In [ ]:
# Extract data for each record set into DataFrames using their @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Columns found: {df.columns.tolist()}")
        display(df.head())
    else:
        print("  [No data available]")
    print()
if len(dataframes) == 0:
    print("No tabular data found via record sets.")
# For further steps, select a representative record set (here, the first one if exists)
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    print("Selected record set for detailed analysis:", selected_record_set_id)
    print("Sample columns:", dataframes[selected_record_set_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)

In this section, we demonstrate filtering and transforming numeric data fields. All columns are referenced using their Croissant field/column `@id`.

*Example: We select a numeric field (e.g., related to protein abundance or molecular weight), filter by a threshold, normalize, and group by another field, if available.*

In [ ]:
import numpy as np
# Choose a numeric field for demonstration, using column/@id as key (update as appropriate)
# For example, let's try to find a numeric column, e.g. 'cr:MolecularWeight', 'cr:MW' or similar.
if len(dataframes) == 0:
    print("No data available for EDA.")
else:
    df = dataframes[selected_record_set_id]
    # Try to guess a numeric field from the columns
    candidate_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    if not candidate_fields:
        # Try to coerce some fields to numeric
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col].dropna().iloc[0])
                candidate_fields.append(col)
            except (ValueError, TypeError, IndexError):
                continue
    if candidate_fields:
        numeric_field = candidate_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # Ensure it's numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75) # 75th percentile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):")
        display(filtered_df[[numeric_field]].head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by a categorical field, e.g. condition or sample
        # If there is a field with <10 unique values, use it
        group_field = None
        for col in df.columns:
            nunique = df[col].nunique()
            dtype = df[col].dtype
            if dtype == object and 1 < nunique < 10 and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"Grouped by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
            display(grouped_df)
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields found in DataFrame columns:", df.columns.tolist())

## 5. Visualization

Here are simple visualizations of the dataset, using the DataFrame loaded from Croissant, with fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion

This notebook demonstrated how to load, explore, and process a FAIR^2 Croissant dataset using its schema URL and the `mlcroissant` library. All entities such as record sets and fields are referenced by their Croissant `@id`. You can extend these steps for further domain-specific analysis (e.g., proteomics, molecular mass profiles, etc.) depending on the dataset content.